# Baseline Models

This notebook evaluates baseline machine-learning models without class balancing.

The purpose is to establish the unbalanced-data reference performance against which subsequent class-imbalance strategies will be compared.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

## Baseline Models (Steps 18-20)

This section defines the unbalanced-data baselines and the evaluation protocol used to compare them.

- Models: prior-probability dummy, logistic regression, decision tree, random forest and RBF SVM.
- No class weighting, resampling, SMOTE/ADASYN, threshold tuning or calibration.
- Every model is a scikit-learn `Pipeline` (`preprocessor` + `classifier`), so median imputation, scaling and one-hot encoding are fitted inside each cross-validation training fold only.
- The held-out test partition stays locked: it is not used for fitting, model selection or evaluation here.

The cheap end-to-end check is `python -m src.smoke_test`. The full repeated cross-validation experiment is run in a later step.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CV_N_SPLITS, DECISION_THRESHOLD, RANDOM_STATE
from src.models import model_parameter_table

parameter_table = pd.DataFrame(model_parameter_table())
parameter_table[["model", "classifier", "preprocessing_family", "parameters"]]

In [ ]:
from src.models import build_baseline_pipelines, model_family

pipelines = build_baseline_pipelines()

pd.DataFrame(
    {
        "model": list(pipelines),
        "preprocessing_family": [model_family(name) for name in pipelines],
        "classifier": [
            type(pipe.named_steps["classifier"]).__name__
            for pipe in pipelines.values()
        ],
        "cv_folds": CV_N_SPLITS,
        "decision_threshold": DECISION_THRESHOLD,
        "random_state": RANDOM_STATE,
    }
)

## Data access policy

The split is reproduced exactly as in Notebook 02 (`random_state=42`, stratified 20% test set). The test partition is wrapped in a locked container: reading its values raises until a final evaluation is explicitly authorised.

In [ ]:
from src.data_loading import TestSetAccessError, load_split

split = load_split()
print("Training partition:", split.X_train.shape)
print("Held-out partition (locked):", split.test.shape)

try:
    split.test.get()
except TestSetAccessError as exc:
    print("Blocked as expected:", exc)